# Model Context Protocol: Advanced Topics

**Source:** Anthropic Partners (Skilljar) course, *Model Context Protocol: Advanced Topics*

## Table of Contents

1. Lecture 1 - Let's get started!
2. Lecture 2 - Sampling
3. Lecture 3 - Sampling walkthrough
4. Lecture 4 - Log and progress notifications
5. Lecture 5 - Notifications walkthrough
6. Lecture 6 - Roots
7. Lecture 7 - Roots walkthrough
8. Lecture 8 - Survey
9. Lecture 9 - JSON message types
10. Lecture 10 - The STDIO transport
11. Lecture 11 - The StreamableHTTP transport
12. Lecture 12 - StreamableHTTP in depth
13. Lecture 13 - State and the StreamableHTTP transport

---

### The 10-minute version

- **Sampling** lets a *server* borrow the *client's* model access, so the client pays for tokens and the server needs no API key.
- **Logging and progress notifications** are optional UX features delivered through the tool's `Context` object.
- **Roots** grant a server scoped access to specific folders. The SDK does **not** enforce them, you must check paths yourself.
- MCP is **bidirectional**: servers can send requests to clients, not just the reverse.
- **STDIO** gives you full bidirectional communication for free, but only on one machine.
- **StreamableHTTP** recreates that bidirectionality over HTTP using **Server-Sent Events (SSE)** plus an `mcp-session-id`.
- Setting `stateless_http=True` or `json_response=True` **breaks** the SSE workaround, killing sampling, progress and logging. Always test on the transport you will deploy on.

> **Note on the diagrams.** They render in JupyterLab, VS Code and nbviewer. GitHub's notebook preview strips inline SVG, so they will appear blank there.


### Summary - Lecture 1

Short (1m28s) video introduction with no transcript text. It frames the course: after the basics of tools, resources and prompts, this course covers the *advanced* half of MCP, namely sampling, logging and progress notifications, roots, and the transport layer (STDIO vs StreamableHTTP).

# Lecture 1 - Let's get started!

### Course scope

The course is organised into two substantive blocks:

1. **Core MCP features**: sampling, log and progress notifications, roots (each concept lesson is followed by a code walkthrough).
2. **Transports and communication**: JSON message types, the STDIO transport, and the StreamableHTTP transport including its configuration flags.

> Video-only lesson. No transcript content was available to summarise.

### Summary - Lecture 2

**Sampling** lets an MCP server reach a language model *through* the connected client instead of calling the model itself. The server builds a prompt, sends a sampling request, and the client (which already holds credentials and a model connection) makes the call and returns the text.

Why it matters:

- The server needs **no API key** and no model integration code.
- The **client pays** for token usage, not the server.
- This is the difference between a public MCP server being viable and it bankrupting you.

Implementation is two-sided: the server calls `ctx.session.create_message()` inside a tool; the client supplies a `sampling_callback` when constructing its `ClientSession`.

# Lecture 2 - Sampling

### ★ What sampling is

Sampling allows a server to access a language model like Claude **through a connected MCP client**. Instead of the server calling the model directly, it asks the client to make the call on its behalf, shifting both the responsibility and the cost of text generation onto the client.

### The problem it solves

Take an MCP server with a research tool that fetches Wikipedia articles and then needs to summarise them into a report. Two options:

| Option | What it means | Cost |
|---|---|---|
| Direct model access | Server needs its own API key, auth handling, cost management, and full integration code | Server pays, high complexity |
| **Sampling** | Server builds a prompt and asks the client "could you call Claude for me?" | Client pays, low complexity |

### ★ The sampling flow

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 670" width="880" height="491" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>Sampling flow</title>
  <desc>Sequence diagram. The MCP Server sends a sampling request to the MCP Client, the Client calls Claude, Claude returns generated text, and the Client returns the result to the Server. A note records that the Client holds the API key and pays for the tokens.</desc>
  <rect width="1200" height="670" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <line x1="250" y1="150" x2="250" y2="610" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>
  <line x1="600" y1="150" x2="600" y2="610" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>
  <line x1="950" y1="150" x2="950" y2="610" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>

  <rect x="120" y="60" width="260" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="250" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>
  <rect x="470" y="60" width="260" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="600" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>
  <rect x="820" y="60" width="260" height="90" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="950" y="112" font-size="19" font-weight="700" text-anchor="middle">Claude</text>

  <line x1="250" y1="230" x2="592" y2="230" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="255" y="196" width="340" height="27" fill="#FFFFFF"/>
  <text x="425" y="217" font-size="19" font-weight="700" text-anchor="middle">1.  Sampling request</text>

  <line x1="600" y1="310" x2="942" y2="310" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="700" y="286" width="150" height="21" fill="#FFFFFF"/>
  <text x="775" y="302" font-size="16" fill="#5F6368" text-anchor="middle">2.  Call the model</text>

  <line x1="950" y1="370" x2="608" y2="370" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="702" y="346" width="146" height="21" fill="#FFFFFF"/>
  <text x="775" y="362" font-size="16" fill="#5F6368" text-anchor="middle">3.  Generated text</text>

  <line x1="600" y1="430" x2="258" y2="430" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="330" y="406" width="190" height="21" fill="#FFFFFF"/>
  <text x="425" y="422" font-size="16" fill="#5F6368" text-anchor="middle">4.  CreateMessageResult</text>

  <rect x="470" y="470" width="260" height="110" rx="8" fill="#E3F6E3" stroke="#5CBF63" stroke-width="3"/>
  <text x="600" y="518" font-size="19" text-anchor="middle">
    <tspan x="600" dy="0" font-weight="700">Holds the API key</tspan>
    <tspan x="600" dy="28">Pays for the tokens</tspan>
  </text>
</svg>

*Sampling in four messages. The server never holds credentials; the client already has a model connection, so it makes the call and absorbs the token cost.*

1. Server completes its own work (e.g. fetching Wikipedia articles).
2. Server creates a prompt asking for text generation.
3. Server sends a **sampling request** to the client.
4. Client calls Claude with the provided prompt.
5. Client returns the generated text to the server.
6. Server uses that text in its tool response.

### Benefits

- **Reduces server complexity**, no direct language model integration.
- **Shifts the cost burden**, the client pays for tokens.
- **No API keys needed** on the server side.
- **Ideal for public servers**, you do not want anonymous users running up your model bill.

### ★ When to use it

Sampling is most valuable for **publicly accessible MCP servers**. Each client pays for its own usage while still getting the full benefit of your server's functionality. In effect it moves the AI integration complexity out of your server and into the client, which usually already has the connections and credentials in place.

### Implementation - server side

Inside a tool function, use `ctx.session.create_message()` to request text generation from the client.

In [ ]:
# FUNCTION - summarize: asks the CLIENT to run a summarisation prompt through its own model
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
    Please summarize the following text:
    {text_to_summarize}
    """

    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user",
                content=TextContent(
                    type="text",
                    text=prompt
                )
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant",
    )

    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")

### Implementation - client side

The client defines a callback that fulfils the server's sampling requests, then passes it into `ClientSession`.

In [ ]:
# FUNCTION - sampling_callback: fulfils a server's sampling request using the client's own model access
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)

    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )


# Register the callback when creating the session
async with ClientSession(
    read,
    write,
    sampling_callback=sampling_callback
) as session:
    await session.initialize()

### Summary - Lecture 3

Hands-on code walkthrough of the sampling implementation from Lecture 2. Video-only, no transcript text on the page.

**Download:** `sampling.zip`, the authoritative source for the server-side `create_message()` call and the client-side `sampling_callback`.

# Lecture 3 - Sampling walkthrough

### Exercise

Wire sampling end to end: add a tool that calls `ctx.session.create_message()`, then run it against a client that registers a `sampling_callback`. Use the code in `sampling.zip` as the reference implementation (it matches the snippets in Lecture 2).

### Summary - Lecture 4

Logging and progress notifications are cheap to add and make a large difference to perceived quality. Without them, a user who triggers a long-running tool sees nothing until it finishes and cannot tell whether it is working or stalled.

Both are delivered through the `Context` object that the Python SDK injects into your tool function:

- `context.info()` sends log messages to the client.
- `context.report_progress(current, total)` sends progress updates.

On the client side, `logging_callback` is registered **on the session**, while `progress_callback` is passed **per tool call**. Both are entirely optional, they are UX enhancements and the client is free to ignore, filter, or render them however it likes.

# Lecture 4 - Log and progress notifications

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 750" width="880" height="550" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>Logging and progress notification flow</title>
  <desc>Sequence diagram. The MCP Client calls a tool passing a progress callback. While the tool runs, the MCP Server emits interleaved log and progress messages back to the Client, then finally the tool result.</desc>
  <rect width="1200" height="750" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <line x1="350" y1="150" x2="350" y2="560" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>
  <line x1="850" y1="150" x2="850" y2="560" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>

  <rect x="220" y="60" width="260" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="350" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>
  <rect x="720" y="60" width="260" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="850" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>

  <line x1="350" y1="230" x2="842" y2="230" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="415" y="196" width="370" height="27" fill="#FFFFFF"/>
  <text x="600" y="217" font-size="19" font-weight="700" text-anchor="middle">call_tool(  progress_callback=fn  )</text>

  <line x1="850" y1="300" x2="358" y2="300" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="488" y="276" width="224" height="21" fill="#FFFFFF"/>
  <text x="600" y="292" font-size="16" fill="#5F6368" text-anchor="middle">info:  About to do research...</text>

  <line x1="850" y1="350" x2="358" y2="350" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="498" y="326" width="204" height="21" fill="#FFFFFF"/>
  <text x="600" y="342" font-size="16" fill="#5F6368" text-anchor="middle">report_progress(20, 100)</text>

  <line x1="850" y1="400" x2="358" y2="400" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="508" y="376" width="184" height="21" fill="#FFFFFF"/>
  <text x="600" y="392" font-size="16" fill="#5F6368" text-anchor="middle">info:  Writing report...</text>

  <line x1="850" y1="450" x2="358" y2="450" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="498" y="426" width="204" height="21" fill="#FFFFFF"/>
  <text x="600" y="442" font-size="16" fill="#5F6368" text-anchor="middle">report_progress(70, 100)</text>

  <line x1="850" y1="510" x2="358" y2="510" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="548" y="486" width="104" height="21" fill="#FFFFFF"/>
  <text x="600" y="502" font-size="16" fill="#5F6368" text-anchor="middle">Tool result</text>

  <rect x="290" y="600" width="620" height="90" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="600" y="638" font-size="19" text-anchor="middle">
    <tspan x="600" dy="0">logging_callback  registers on the session</tspan>
    <tspan x="600" dy="28" font-weight="700">progress_callback  passes per tool call</tspan>
  </text>
</svg>

*One tool call, five messages back. Logs and progress interleave while the tool is still running, which is what turns a frozen-looking client into a responsive one.*

### Why they exist

When Claude calls a tool that takes time (researching a topic, processing data), the user typically sees nothing until the operation completes. That is frustrating because there is no signal distinguishing "working" from "broken". With notifications enabled, users get real-time feedback: progress bars, status messages, and logs as the operation runs.

### Server side - the Context object

In the Python MCP SDK, both features work through the `Context` argument automatically supplied to your tool function.

| Method | Purpose |
|---|---|
| `context.info()` | Send a log message to the client |
| `context.report_progress(progress, total)` | Update progress with current and total values |

In [ ]:
# FUNCTION - research: long-running tool that streams logs and progress back to the client
@mcp.tool(
    name="research",
    description="Research a given topic"
)
async def research(
    topic: str = Field(description="Topic to research"),
    *,
    context: Context
):
    await context.info("About to do research...")
    await context.report_progress(20, 100)
    sources = await do_research(topic)

    await context.info("Writing report...")
    await context.report_progress(70, 100)
    results = await generate_report(sources)

    return results

### ★ Client side - where each callback is registered

This is the detail that trips people up:

1. The **logging callback** is passed when creating the `ClientSession`.
2. The **progress callback** is passed on each individual `call_tool()`.

That split is deliberate, it lets you handle logs globally but treat progress differently per call.

In [ ]:
# FUNCTION - logging_callback: renders server log notifications for this client
async def logging_callback(params: LoggingMessageNotificationParams):
    print(params.data)


# FUNCTION - print_progress_callback: renders progress updates, tolerating an unknown total
async def print_progress_callback(
    progress: float, total: float | None, message: str | None
):
    if total is not None:
        percentage = (progress / total) * 100
        print(f"Progress: {progress}/{total} ({percentage:.1f}%)")
    else:
        print(f"Progress: {progress}")


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read,
            write,
            logging_callback=logging_callback   # session level
        ) as session:
            await session.initialize()

            await session.call_tool(
                name="add",
                arguments={"a": 1, "b": 3},
                progress_callback=print_progress_callback,   # per call
            )

### Presentation options by application type

| Application type | How to surface notifications |
|---|---|
| CLI | Print messages and progress straight to the terminal |
| Web | WebSockets, server-sent events, or polling to push updates to the browser |
| Desktop | Update native progress bars and status displays |

Implementing these is **entirely optional**. A client may ignore them completely, show only certain types, or present them in whatever way fits the application.

### Summary - Lecture 5

Code walkthrough of the logging and progress implementation from Lecture 4. Video-only, no transcript text on the page.

**Download:** `notifications.zip`, reference code for the server-side `context.info()` and `context.report_progress()` calls plus the client-side `logging_callback` and `progress_callback`.

# Lecture 5 - Notifications walkthrough

### Exercise

Add `context.info()` and `context.report_progress()` to a slow tool, then register both callbacks on the client and confirm the messages arrive in the right order while the tool is still running. Reference code is in `notifications.zip`.

### Summary - Lecture 6

**Roots** grant an MCP server scoped access to specific files and folders on the local machine. They serve two purposes at once: a permission boundary, and the context Claude needs to *find* a file the user referred to by name only.

Without roots, a user asking to "convert biking.mp4" gives Claude a bare filename with no way to locate it. With roots, Claude calls `list_roots()`, then `read_dir()` on the permitted directories, finds the full path, and calls the tool with it.

**★ The critical caveat:** the MCP SDK does **not** enforce root restrictions for you. You must write your own `is_path_allowed()` check and call it in every tool that touches the filesystem.

# Lecture 6 - Roots

### What roots are

Roots are a way to tell an MCP server "you may access these files and folders". Think of them as a permission system, but they do more than grant permission, they also give Claude the search context it needs.

### The problem they solve

An MCP server exposes a video conversion tool that takes a file path and converts MP4 to MOV. The user asks Claude to "convert biking.mp4 to mov format".

- Claude receives only the **filename**, not a path.
- Claude cannot search the entire file system to find it.
- The user knows it is in their Movies folder, Claude does not.

You could demand full paths from users, but nobody wants to type complete paths every time.

### ★ Roots in action

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 610" width="860" height="610" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>How roots resolve a bare filename</title>
  <desc>Four step flow. The user names a file with no path, Claude calls list_roots to see permitted directories, calls read_dir to locate the file, then calls the tool with the full resolved path.</desc>
  <rect width="860" height="610" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <text x="95" y="84" font-size="46" font-weight="700">1</text>
  <rect x="60" y="100" width="320" height="150" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="220" y="154" font-size="19" text-anchor="middle">
    <tspan x="220" dy="0">User asks to convert</tspan>
    <tspan x="220" dy="28" font-weight="700">biking.mp4</tspan>
    <tspan x="220" dy="28">with no path given</tspan>
  </text>

  <line x1="380" y1="175" x2="472" y2="175" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>

  <text x="515" y="84" font-size="46" font-weight="700">2</text>
  <rect x="480" y="100" width="320" height="150" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="640" y="154" font-size="19" text-anchor="middle">
    <tspan x="640" dy="0">Claude calls</tspan>
    <tspan x="640" dy="28" font-weight="700">list_roots()</tspan>
    <tspan x="640" dy="28">to see permitted dirs</tspan>
  </text>

  <path d="M 640 250 L 640 325 L 220 325 L 220 392" fill="none" stroke="#000000" stroke-width="7"
        stroke-linejoin="round" marker-end="url(#ah)"/>

  <text x="95" y="384" font-size="46" font-weight="700">3</text>
  <rect x="60" y="400" width="320" height="150" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="220" y="454" font-size="19" text-anchor="middle">
    <tspan x="220" dy="0">Claude calls</tspan>
    <tspan x="220" dy="28" font-weight="700">read_dir()</tspan>
    <tspan x="220" dy="28">to locate the file</tspan>
  </text>

  <line x1="380" y1="475" x2="472" y2="475" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>

  <text x="515" y="384" font-size="46" font-weight="700">4</text>
  <rect x="480" y="400" width="320" height="150" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/>
  <text x="640" y="468" font-size="19" text-anchor="middle">
    <tspan x="640" dy="0">Claude calls the tool</tspan>
    <tspan x="640" dy="28" font-weight="700">with the full path</tspan>
  </text>
</svg>

*Roots do double duty: they bound what the server may touch, and they give Claude a short list of places to look so a bare filename can be resolved automatically.*

All of this happens automatically, so the user can still just say "convert biking.mp4".

### Security and boundaries

Roots also limit access. If you grant only the Desktop folder, the server cannot reach Documents or Downloads. When Claude attempts a file outside the approved roots, it gets an error and can tell the user the file is not accessible under the current server configuration.

### ★ Implementation - you must enforce roots yourself

The SDK does **not** automatically enforce root restrictions. The standard pattern is a helper that:

1. Takes a requested file path.
2. Gets the list of approved roots.
3. Checks whether the requested path falls within one of them.
4. Returns true/false for access permission.

Call this helper in **every** tool that reads a file or directory, before performing the actual operation.

In [ ]:
# FUNCTION - is_path_allowed: returns True only if requested_path falls inside one of the approved roots
async def is_path_allowed(requested_path: str, ctx: Context) -> bool:
    roots = await ctx.session.list_roots()
    # Compare the resolved requested path against each approved root
    # Return True on the first match, False if none match
    ...


@mcp.tool()
async def convert_video(path: str, ctx: Context):
    if not await is_path_allowed(path, ctx):
        raise ValueError("Path is outside the approved roots")
    # ... perform the conversion

### Key benefits

- **User friendly**, users never have to supply full file paths.
- **Focused search**, Claude only looks in approved directories, so discovery is faster.
- **Security**, prevents accidental access to sensitive files outside approved areas.
- **Flexibility**, roots can be provided through tools or injected directly into prompts.

### Summary - Lecture 7

Code walkthrough of the roots implementation from Lecture 6. Video-only, no transcript text on the page.

**Download:** `roots.zip`, reference code for declaring roots on the client and enforcing them with an `is_path_allowed()` style helper on the server.

# Lecture 7 - Roots walkthrough

### Exercise

Configure a client with a single root, then build a tool that calls `list_roots()`, resolves a bare filename to a full path, and rejects any path outside the root. Reference code is in `roots.zip`.

### Summary - Lecture 8

Course satisfaction survey for the Advanced MCP course, 3 questions. No instructional content to summarise.

# Lecture 8 - Survey

Optional 3-question course satisfaction survey. Nothing to take notes on.

### Summary - Lecture 9

All MCP communication is JSON messages, and they fall into exactly two shapes:

1. **Request and Result pairs** (Call Tool Request with Call Tool Result, Initialize Request with Initialize Result, and so on).
2. **Notifications**, one-way messages that expect no response (progress, logging, tool list changed, resource updated).

The specification also organises messages by **who sends them**, client or server. The takeaway that matters later: **MCP is bidirectional**, servers can initiate messages to clients, not just respond. Some transports (notably StreamableHTTP) restrict which directions are actually available, which is why this classification matters.

The authoritative list lives in the **MCP specification repository on GitHub**, separate from the language SDK repos. It is written in TypeScript purely as a convenient way to describe data structures, not because it executes.

# Lecture 9 - JSON message types

### Message format

Every MCP interaction happens through JSON messages, each type serving a specific purpose: calling a tool, listing resources, or reporting a system event. Typical exchange: Claude needs to run a server tool, so the client sends a **Call Tool Request**; the server runs the tool and replies with a **Call Tool Result** containing the output.

### Where the spec lives

The complete list of message types is defined in the official **MCP specification repository** on GitHub. This is separate from the SDK repositories (Python, TypeScript, etc.) and is the authoritative source for how MCP should work. The types are written in TypeScript for convenience, because TypeScript describes data structures clearly, not because they are executed as code.

### ★ The two message categories

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 650" width="880" height="572" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>The two MCP message categories</title>
  <desc>Two grouped panels. The upper panel shows a request and result always travelling as a pair between client and server. The lower panel shows a notification travelling one way from server to client with no response.</desc>
  <rect width="1000" height="650" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <rect x="60" y="70" width="880" height="240" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/>
  <text x="78" y="96" font-size="16" fill="#5F6368">Request and Result  (always paired)</text>

  <rect x="110" y="140" width="240" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="230" y="192" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>
  <rect x="650" y="140" width="240" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="770" y="192" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>

  <line x1="350" y1="170" x2="642" y2="170" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="424" y="141" width="144" height="21" fill="#FFFFFF"/>
  <text x="496" y="157" font-size="16" fill="#5F6368" text-anchor="middle">Call Tool Request</text>

  <line x1="650" y1="215" x2="358" y2="215" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="429" y="186" width="134" height="21" fill="#FFFFFF"/>
  <text x="496" y="202" font-size="16" fill="#5F6368" text-anchor="middle">Call Tool Result</text>

  <rect x="60" y="350" width="880" height="240" rx="8" fill="none" stroke="#8B8BF5" stroke-width="2" stroke-dasharray="8 6"/>
  <text x="78" y="376" font-size="16" fill="#5F6368">Notification  (one way, no response)</text>

  <rect x="110" y="420" width="240" height="90" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="230" y="472" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>
  <rect x="650" y="420" width="240" height="90" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/>
  <text x="770" y="472" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>

  <line x1="350" y1="465" x2="642" y2="465" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="409" y="436" width="174" height="21" fill="#FFFFFF"/>
  <text x="496" y="452" font-size="16" fill="#5F6368" text-anchor="middle">Progress Notification</text>
</svg>

*Every MCP message is one of these two shapes. Whether a message expects a reply determines which transports can carry it, which is the whole subject of the next section.*

**Request and Result messages** always come in pairs, you send a request and expect a result:

- Call Tool Request with Call Tool Result
- List Prompts Request with List Prompts Result
- Read Resource Request with Read Resource Result
- Initialize Request with Initialize Result

**Notification messages** are one-way and require no response:

- Progress Notification, updates on long-running operations
- Logging Message Notification, system log messages
- Tool List Changed Notification, available tools changed
- Resource Updated Notification, resources were modified

### Client vs server messages

The spec organises messages by sender:

- **Client messages**: requests clients send to servers (such as tool calls) plus client notifications.
- **Server messages**: requests servers send to clients plus server notifications.

### ★ Why this matters

The key insight is that **MCP is a bidirectional protocol**, both sides can initiate communication. That becomes critical when choosing a transport, because some transports (StreamableHTTP in particular) limit which message types can flow in which direction.

### Summary - Lecture 10

A **transport** is the channel that actually carries MCP's JSON messages. During development, the usual choice is **STDIO**: the client launches the server as a subprocess and they talk over stdin/stdout.

Every MCP connection opens with a fixed **three-message handshake**: Initialize Request, Initialize Result, Initialized Notification. Only then can tool calls or listings be sent.

STDIO's value is that it is the *ideal* case, either side can send at any time, so all four communication patterns work without effort. Its limit is that both parties must run on the same machine. It is also directly testable, run `uv run server.py` and paste JSON straight into the terminal.

Treat STDIO as the baseline for what "full MCP communication" looks like before dealing with HTTP's constraints.

# Lecture 10 - The STDIO transport

### What a transport is

Clients and servers exchange JSON messages, but something has to actually transmit them. That channel is the **transport**. Implementations range from HTTP requests to WebSockets (and in principle anything that moves bytes, though only some are production-sensible).

### How STDIO works

The client launches the MCP server as a **subprocess** and communicates over the standard streams:

- Client sends messages to the server via the server's **stdin**.
- Server responds by writing to **stdout**.
- Either side can send a message at any time.
- Only works when client and server run on the **same machine**.

### Testing it without writing a client

Run the server with `uv run server.py`. It listens on stdin and writes to stdout, so you can paste JSON messages directly into the terminal and read the responses immediately, including the full initialization and tool call exchange.

In [ ]:
# Run the server directly and speak JSON to it over stdin
uv run server.py

### ★ The MCP connection sequence

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 792" width="880" height="581" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>The STDIO connection handshake</title>
  <desc>Sequence diagram. A dashed group marks the mandatory three message handshake: initialize request, initialize result, initialized notification. Only afterwards can a tool call request and result be exchanged.</desc>
  <rect width="1200" height="792" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <line x1="350" y1="150" x2="350" y2="560" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>
  <line x1="850" y1="150" x2="850" y2="560" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>

  <rect x="220" y="60" width="260" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="350" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>
  <rect x="720" y="60" width="260" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="850" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>

  <rect x="170" y="185" width="860" height="215" rx="8" fill="none" stroke="#6BB8F5" stroke-width="2" stroke-dasharray="8 6"/>
  <rect x="182" y="195" width="415" height="21" fill="#FFFFFF"/>
  <text x="188" y="211" font-size="16" fill="#5F6368">Mandatory handshake, before anything else may be sent</text>

  <line x1="350" y1="260" x2="842" y2="260" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="515" y="236" width="170" height="21" fill="#FFFFFF"/>
  <text x="600" y="252" font-size="16" fill="#5F6368" text-anchor="middle">1.  Initialize Request</text>

  <line x1="850" y1="315" x2="358" y2="315" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="522" y="291" width="156" height="21" fill="#FFFFFF"/>
  <text x="600" y="307" font-size="16" fill="#5F6368" text-anchor="middle">2.  Initialize Result</text>

  <line x1="350" y1="370" x2="842" y2="370" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="450" y="346" width="300" height="21" fill="#FFFFFF"/>
  <text x="600" y="362" font-size="16" fill="#5F6368" text-anchor="middle">3.  Initialized Notification  (no response)</text>

  <line x1="350" y1="450" x2="842" y2="450" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="505" y="417" width="190" height="27" fill="#FFFFFF"/>
  <text x="600" y="438" font-size="19" font-weight="700" text-anchor="middle">Call Tool Request</text>

  <line x1="850" y1="510" x2="358" y2="510" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="527" y="486" width="146" height="21" fill="#FFFFFF"/>
  <text x="600" y="502" font-size="16" fill="#5F6368" text-anchor="middle">Call Tool Result</text>

  <rect x="180" y="600" width="840" height="132" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/>
  <text x="600" y="645" font-size="19" text-anchor="middle">
    <tspan x="600" dy="0">Client writes to the server's stdin.  Server writes to stdout.</tspan>
    <tspan x="600" dy="28">Either side may send at any time,</tspan>
    <tspan x="600" dy="28" font-weight="700">but only after the handshake.</tspan>
  </text>
</svg>

*The three-message handshake is mandatory and ordered. Nothing else may be sent until the Initialized Notification lands.*

1. **Initialize Request**, sent by the client first.
2. **Initialize Result**, server responds with its capabilities.
3. **Initialized Notification**, client confirms (no response expected).

Only after the handshake completes can you send other requests such as tool calls or prompt listings.

### ★ The four communication scenarios

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1120 630" width="880" height="495" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>How STDIO carries all four communication patterns</title>
  <desc>The client launches the server as a subprocess. Two channels connect them: stdin carries everything the client sends, stdout carries everything the server sends. Because either side may write at any time, both requests and responses ride each channel.</desc>
  <rect width="1120" height="630" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <rect x="60" y="210" width="300" height="140" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="210" y="287" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>

  <rect x="760" y="210" width="300" height="140" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="910" y="273" font-size="19" text-anchor="middle">
    <tspan x="910" dy="0" font-weight="700">MCP Server</tspan>
    <tspan x="910" dy="28">(a subprocess)</tspan>
  </text>

  <line x1="360" y1="245" x2="752" y2="245" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="524" y="211" width="72" height="27" fill="#FFFFFF"/>
  <text x="560" y="232" font-size="19" font-weight="700" text-anchor="middle">stdin</text>

  <line x1="760" y1="315" x2="368" y2="315" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="516" y="281" width="88" height="27" fill="#FFFFFF"/>
  <text x="560" y="302" font-size="19" font-weight="700" text-anchor="middle">stdout</text>

  <rect x="60" y="420" width="460" height="150" rx="8" fill="#ECF3FC" stroke="#6BB8F5" stroke-width="3"/>
  <text x="290" y="474" font-size="19" text-anchor="middle">
    <tspan x="290" dy="0" font-weight="700">Everything on stdin</tspan>
    <tspan x="290" dy="28">Client to Server requests</tspan>
    <tspan x="290" dy="28">Client to Server responses</tspan>
  </text>

  <rect x="600" y="420" width="460" height="150" rx="8" fill="#FCEDF5" stroke="#FF6EC7" stroke-width="3"/>
  <text x="830" y="474" font-size="19" text-anchor="middle">
    <tspan x="830" dy="0" font-weight="700">Everything on stdout</tspan>
    <tspan x="830" dy="28">Server to Client responses</tspan>
    <tspan x="830" dy="28">Server to Client requests</tspan>
  </text>
</svg>

*Two channels, four patterns. Because either side may write whenever it likes, requests and responses both ride each stream. This is the freedom that HTTP takes away.*

Any transport has to handle all four of these patterns. Under STDIO each maps onto one of the two streams:

| # | Pattern | STDIO mechanism |
|---|---|---|
| 1 | Client to Server request | Client writes to stdin |
| 2 | Server to Client response | Server writes to stdout |
| 3 | Server to Client request | Server writes to stdout |
| 4 | Client to Server response | Client writes to stdin |

The elegance of STDIO is that scenarios 3 and 4 come for free, either party can initiate at any time using just those two channels.

### Why this is the baseline

STDIO represents the ideal case where bidirectional communication is seamless. Other transports, HTTP especially, cannot always let the server initiate requests to the client. Understanding STDIO first gives you a picture of complete MCP communication before you deal with the constraints of everything else.

**Rule of thumb:** STDIO for development and testing; a different transport once client and server need to run on separate machines.

### Summary - Lecture 11

**StreamableHTTP** lets clients connect to remotely hosted MCP servers over HTTP, which is what makes public MCP servers possible. The catch is two configuration flags:

- `stateless_http`
- `json_response`

Both default to `False`. Some deployment scenarios force you to set them `True`, and when you do, you break core functionality: progress notifications, logging, and server-initiated requests all stop working.

**★ The diagnostic to remember:** if your server works perfectly under STDIO locally but breaks once deployed over HTTP, these flags are the likely culprit.

The underlying cause is structural. HTTP assumes clients have no known URL, so servers cannot easily initiate requests to them. That directly hits the message types MCP depends on: Create Message (sampling) requests, List Roots requests, and every kind of notification.

# Lecture 11 - The StreamableHTTP transport

### What it enables

The StreamableHTTP transport lets MCP clients connect to **remotely hosted servers** over HTTP. Unlike STDIO, which requires both sides on the same machine, this opens the door to public MCP servers that anyone can reach.

### ★ The two flags that matter

| Setting | Controls | Default |
|---|---|---|
| `stateless_http` | Connection state management | `False` |
| `json_response` | Response format handling | `False` |

Certain deployment scenarios force these to `True`. Once enabled, they can break progress notifications, logging, and server-initiated requests.

> **Diagnostic:** works under STDIO locally but breaks over HTTP after deployment, check these two flags first.

### The underlying HTTP problem

In standard HTTP:

- Clients can easily initiate requests to servers, the server has a known URL.
- Servers can easily respond to those requests.
- **Servers cannot easily initiate requests to clients**, clients have no known URL.
- Response patterns flowing from client back to server become awkward.

### ★ Which MCP messages this breaks

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 620" width="880" height="455" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>What the StreamableHTTP flags break</title>
  <desc>Two panels. With the default flags the server can initiate requests and notifications to the client. With stateless_http or json_response set to true, that whole direction is blocked, taking sampling, progress reports and logging with it.</desc>
  <rect width="1200" height="620" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <rect x="60" y="70" width="1080" height="210" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/>
  <text x="78" y="96" font-size="16" fill="#5F6368">Default:  stateless_http=False,  json_response=False</text>

  <rect x="140" y="150" width="260" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="270" y="202" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>
  <rect x="800" y="150" width="260" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="930" y="202" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>

  <line x1="400" y1="195" x2="792" y2="195" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="436" y="166" width="320" height="21" fill="#FFFFFF"/>
  <text x="596" y="182" font-size="16" fill="#5F6368" text-anchor="middle">Server-initiated requests and notifications</text>

  <rect x="60" y="350" width="1080" height="210" rx="8" fill="none" stroke="#FA6E6E" stroke-width="2" stroke-dasharray="8 6"/>
  <text x="78" y="376" font-size="16" fill="#5F6368">With  stateless_http=True  or  json_response=True</text>

  <rect x="140" y="430" width="260" height="90" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="270" y="482" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>
  <rect x="800" y="430" width="260" height="90" rx="8" fill="#FCEDF5" stroke="#D6009A" stroke-width="3"/>
  <text x="930" y="482" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>

  <line x1="400" y1="475" x2="556" y2="475" stroke="#DC0A0A" stroke-width="7" stroke-dasharray="12 9"/>
  <line x1="578" y1="453" x2="618" y2="497" stroke="#DC0A0A" stroke-width="6"/>
  <line x1="618" y1="453" x2="578" y2="497" stroke="#DC0A0A" stroke-width="6"/>
  <rect x="431" y="506" width="330" height="21" fill="#FFFFFF"/>
  <text x="596" y="522" font-size="16" font-weight="700" fill="#DC0A0A" text-anchor="middle">No sampling, progress or logging</text>
</svg>

*The flags do not degrade the server-to-client direction, they remove it. Everything that depended on the server speaking first goes with it.*

**Server-initiated requests**

- Create Message requests (sampling)
- List Roots requests

**Notifications**

- Progress notifications
- Logging notifications
- Initialized notifications
- Cancelled notifications

These are exactly the features that stop working when the restrictive HTTP settings are enabled: progress bars disappear, logging goes silent, and server-initiated sampling requests fail.

### The trade-off

StreamableHTTP does provide a clever workaround for HTTP's limitations (covered in Lecture 12). When you are forced into `stateless_http=True` or `json_response=True`, you are telling the transport to operate *within* HTTP's constraints rather than work around them.

Knowing this shapes three decisions:

1. Which transport to use for a given deployment scenario.
2. How to design the server to degrade gracefully under HTTP constraints.
3. When reduced functionality is an acceptable price for remote hosting.

If your application leans heavily on server-initiated requests or real-time notifications, either reconsider the transport or design alternative communication patterns.

### Summary - Lecture 12

StreamableHTTP works around HTTP's one-directional nature using **Server-Sent Events (SSE)**.

1. Client sends an Initialize Request.
2. Server responds with an Initialize Result carrying an **`mcp-session-id` header**.
3. Client sends an Initialized Notification with that session ID.
4. Client makes a **GET** request to open a long-lived SSE connection the server can push through at any time.

**★ The dual-connection model** is the part to remember. Every tool call produces a *second*, tool-specific SSE connection:

| Connection | Lifetime | Carries |
|---|---|---|
| **Primary SSE** (GET) | Open indefinitely | Server-initiated requests, **progress notifications** |
| **Tool-specific SSE** | Closes when the tool result is sent | **Logging messages**, tool results |

The session ID is required on all requests after initialization. Setting `stateless_http` or `json_response` to `True` breaks this workaround.

# Lecture 12 - StreamableHTTP in depth

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1300 890" width="880" height="602" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>The StreamableHTTP dual SSE model</title>
  <desc>Sequence diagram. After initialization returns an mcp-session-id, a GET opens a primary SSE channel that stays open and carries server-initiated requests and progress notifications. Each tool call opens a second, tool-specific SSE channel that carries logging messages and the tool result, then closes.</desc>
  <rect width="1300" height="890" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker></defs>

  <line x1="350" y1="150" x2="350" y2="830" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>
  <line x1="950" y1="150" x2="950" y2="830" stroke="#9AA0A6" stroke-width="3" stroke-dasharray="10 8"/>

  <rect x="220" y="60" width="260" height="90" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="350" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>
  <rect x="820" y="60" width="260" height="90" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="950" y="112" font-size="19" font-weight="700" text-anchor="middle">MCP Server</text>

  <line x1="350" y1="230" x2="942" y2="230" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="543" y="206" width="214" height="21" fill="#FFFFFF"/>
  <text x="650" y="222" font-size="16" fill="#5F6368" text-anchor="middle">POST   Initialize Request</text>

  <line x1="950" y1="290" x2="358" y2="290" stroke="#000000" stroke-width="7" marker-end="url(#ah)"/>
  <rect x="480" y="257" width="340" height="27" fill="#FFFFFF"/>
  <text x="650" y="278" font-size="19" font-weight="700" text-anchor="middle">Initialize Result  +  mcp-session-id</text>

  <line x1="350" y1="350" x2="942" y2="350" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="525" y="326" width="250" height="21" fill="#FFFFFF"/>
  <text x="650" y="342" font-size="16" fill="#5F6368" text-anchor="middle">POST   Initialized Notification</text>

  <line x1="350" y1="410" x2="942" y2="410" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="497" y="386" width="306" height="21" fill="#FFFFFF"/>
  <text x="650" y="402" font-size="16" fill="#5F6368" text-anchor="middle">GET   (opens the primary SSE channel)</text>

  <rect x="170" y="445" width="1010" height="145" rx="8" fill="none" stroke="#6BB8F5" stroke-width="2" stroke-dasharray="8 6"/>
  <rect x="182" y="455" width="302" height="21" fill="#FFFFFF"/>
  <text x="188" y="471" font-size="16" fill="#5F6368">PRIMARY SSE   (stays open indefinitely)</text>

  <line x1="950" y1="520" x2="358" y2="520" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="470" y="496" width="360" height="21" fill="#FFFFFF"/>
  <text x="650" y="512" font-size="16" fill="#5F6368" text-anchor="middle">Server-initiated requests  (sampling, list_roots)</text>

  <line x1="950" y1="568" x2="358" y2="568" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="558" y="544" width="184" height="21" fill="#FFFFFF"/>
  <text x="650" y="560" font-size="16" fill="#5F6368" text-anchor="middle">Progress notifications</text>

  <line x1="350" y1="630" x2="942" y2="630" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="545" y="606" width="210" height="21" fill="#FFFFFF"/>
  <text x="650" y="622" font-size="16" fill="#5F6368" text-anchor="middle">POST   Call Tool Request</text>

  <rect x="170" y="665" width="1010" height="150" rx="8" fill="none" stroke="#FF6EC7" stroke-width="2" stroke-dasharray="8 6"/>
  <rect x="182" y="675" width="392" height="21" fill="#FFFFFF"/>
  <text x="188" y="691" font-size="16" fill="#5F6368">TOOL-SPECIFIC SSE   (closes when the result is sent)</text>

  <line x1="950" y1="735" x2="358" y2="735" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="576" y="711" width="148" height="21" fill="#FFFFFF"/>
  <text x="650" y="727" font-size="16" fill="#5F6368" text-anchor="middle">Logging messages</text>

  <line x1="950" y1="785" x2="358" y2="785" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="581" y="761" width="138" height="21" fill="#FFFFFF"/>
  <text x="650" y="777" font-size="16" fill="#5F6368" text-anchor="middle">Call Tool Result</text>
</svg>

*The whole workaround in one picture. Note the split routing: progress notifications go down the primary channel, logging messages go down the tool-specific one. That distinction is what you reason about when a message goes missing.*

### The core problem

Sampling, notifications and logging all rely on the **server initiating requests to the client**. HTTP is built for the opposite direction. StreamableHTTP resolves this with a workaround built on **Server-Sent Events (SSE)**.

### ★ Initial connection setup

1. Client sends an **Initialize Request** to the server.
2. Server responds with an **Initialize Result** that includes a special **`mcp-session-id`** header.
3. Client sends an **Initialized Notification** carrying that session ID.

The session ID uniquely identifies the client and **must be included in every subsequent request**.

### The SSE workaround

After initialization, the client makes a **GET request** to establish a Server-Sent Events connection. This produces a long-lived HTTP response the server can stream messages down at any time, which is what restores server-to-client communication: requests, notifications, and everything else.

### ★ Dual SSE connections during a tool call

| Connection | Lifetime | What routes through it |
|---|---|---|
| **Primary SSE** | Stays open indefinitely | Server-initiated requests, **progress notifications** |
| **Tool-specific SSE** | Created per tool call, closes when the result is sent | **Logging messages**, tool results |

### The flags that break it

`stateless_http` and `json_response` set to `True` can break the SSE workaround. You may want them in certain scenarios, but doing so limits the MCP functionality that depends on server-to-client communication.

### Key takeaways

1. StreamableHTTP is the most complex MCP transport because it has to work around HTTP itself.
2. The SSE workaround enables full MCP functionality over HTTP.
3. Session IDs are required on all requests after initialization.
4. The dual-connection model is essential for debugging and optimisation.

### Summary - Lecture 13

This lecture explains *why* you would ever set the two damaging flags.

**`stateless_http=True`** exists for **horizontal scaling**. Behind a load balancer, a client's GET SSE connection and its POST tool calls can land on different server instances, and coordinating between them is a hard problem. Going stateless eliminates the coordination problem by eliminating the state, at a steep cost: no session IDs, no server-to-client requests, **no sampling**, no progress reports, no subscriptions. The one upside is that client initialization is no longer required.

**`json_response=True`** is simpler, it just disables streaming for POST responses. You get the final tool result as plain JSON, with no intermediate progress messages or log statements.

**★ Practical rule:** if you develop on STDIO but deploy over HTTP, test on the transport you will actually deploy on. The behavioural differences between stateful and stateless modes are significant and much cheaper to find before deployment.

**Download:** `transport-http.zip`

# Lecture 13 - State and the StreamableHTTP transport

### ★ Why stateless HTTP exists, the scaling story

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 720" width="880" height="528" font-family="Helvetica Neue, Helvetica, Arial, sans-serif" role="img">
  <title>Why stateless HTTP exists</title>
  <desc>Architecture diagram. A client opens a GET SSE connection and sends POST tool calls through a load balancer into a pool of server instances. The two connections can land on different instances, which then have to coordinate. A caption explains that stateless HTTP removes the coordination problem by removing the state.</desc>
  <rect width="1200" height="720" fill="#FFFFFF"/>
  <defs><marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#000000"/></marker>
  <marker id="ahr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#DC0A0A"/></marker></defs>

  <rect x="60" y="250" width="240" height="110" rx="8" fill="#F7EDFC" stroke="#A81FEE" stroke-width="3"/>
  <text x="180" y="312" font-size="19" font-weight="700" text-anchor="middle">MCP Client</text>

  <rect x="480" y="250" width="240" height="110" rx="8" fill="#E3F6E3" stroke="#018001" stroke-width="3"/>
  <text x="600" y="312" font-size="19" font-weight="700" text-anchor="middle">Load Balancer</text>

  <rect x="800" y="100" width="340" height="370" rx="8" fill="none" stroke="#00C9A7" stroke-width="2" stroke-dasharray="8 6"/>
  <text x="818" y="126" font-size="16" fill="#5F6368">Server pool</text>

  <rect x="840" y="150" width="260" height="110" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="970" y="212" font-size="19" font-weight="700" text-anchor="middle">Server instance A</text>
  <rect x="840" y="330" width="260" height="110" rx="8" fill="#ECF3FC" stroke="#0B6BE8" stroke-width="3"/>
  <text x="970" y="392" font-size="19" font-weight="700" text-anchor="middle">Server instance B</text>

  <line x1="300" y1="285" x2="472" y2="285" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="346" y="261" width="80" height="21" fill="#FFFFFF"/>
  <text x="386" y="277" font-size="16" fill="#5F6368" text-anchor="middle">GET  SSE</text>

  <line x1="300" y1="335" x2="472" y2="335" stroke="#000000" stroke-width="3" marker-end="url(#ah)"/>
  <rect x="321" y="311" width="130" height="21" fill="#FFFFFF"/>
  <text x="386" y="327" font-size="16" fill="#5F6368" text-anchor="middle">POST  tool calls</text>

  <path d="M 720 285 L 770 285 L 770 205 L 832 205" fill="none" stroke="#000000" stroke-width="3"
        stroke-linejoin="round" marker-end="url(#ah)"/>
  <path d="M 720 335 L 770 335 L 770 385 L 832 385" fill="none" stroke="#000000" stroke-width="3"
        stroke-linejoin="round" marker-end="url(#ah)"/>

  <line x1="970" y1="268" x2="970" y2="322" stroke="#DC0A0A" stroke-width="3" stroke-dasharray="8 6"
        marker-start="url(#ahr)" marker-end="url(#ahr)"/>
  <text x="948" y="301" font-size="16" font-weight="700" fill="#DC0A0A" text-anchor="end">Must coordinate</text>

  <rect x="290" y="530" width="620" height="130" rx="8" fill="#EEEEFC" stroke="#5A4FF0" stroke-width="3"/>
  <text x="600" y="574" font-size="19" text-anchor="middle">
    <tspan x="600" dy="0">stateless_http=True removes this problem</tspan>
    <tspan x="600" dy="28" font-weight="700">by removing the state</tspan>
    <tspan x="600" dy="28">and with it sampling, progress and subscriptions.</tspan>
  </text>
</svg>

*The coordination problem in red. A client's two connections can land on different instances, and sampling needs them to talk to each other. Going stateless dissolves the problem by dissolving the state.*

1. You build an MCP server and a handful of clients connect to a single instance.
2. It becomes popular, thousands of clients try to connect, one instance no longer copes.
3. The standard fix is **horizontal scaling**: multiple server instances behind a load balancer.
4. Here is the complication. Each MCP client needs **two separate connections**:
   - a **GET SSE** connection for receiving server-to-client requests, and
   - **POST** requests for calling tools and receiving responses.
5. The load balancer may route these to **different instances**. If a tool needs sampling, the instance handling the POST must coordinate with the instance holding the GET SSE connection, a genuinely hard distributed-systems problem.

### ★ What `stateless_http=True` costs you

Setting it eliminates the coordination problem, because there is no longer any state to coordinate. The trade-offs:

| Consequence | Effect |
|---|---|
| No session IDs | The server cannot track individual clients |
| No server-to-client requests | The GET SSE pathway becomes unavailable |
| **No sampling** | Cannot use Claude or other models from within a tool |
| No progress reports | No progress updates during long operations |
| No subscriptions | Cannot notify clients about resource updates |
| *(benefit)* | **Client initialization is no longer required**, clients can make requests directly, skipping the handshake |

### What `json_response=True` costs you

Simpler in scope: it disables **streaming** for POST responses. Instead of a series of SSE messages as the tool executes, you receive only the final result as plain JSON.

- No intermediate progress messages
- No log statements during execution
- Just the final tool result

### Decision guide

**Use `stateless_http` when:**

- You need horizontal scaling behind load balancers.
- You do not need server-to-client communication.
- Your tools do not require model sampling.
- You want to minimise connection overhead.

**Use `json_response` when:**

- You do not need streaming responses.
- You prefer simpler, non-streaming HTTP responses.
- You are integrating with systems that expect plain JSON.

### ★ Development vs production

If you develop locally on STDIO but plan to deploy over HTTP, **test with the transport you will deploy on**. The behavioural differences between stateful and stateless modes are significant, and far cheaper to discover during development than after deployment.

**Download:** `transport-http.zip`, reference code for the StreamableHTTP transport and both configuration flags.